# Build the DIZON base model

This is the first notebook of **part1** — the DIZON decision-support workflow proper. Here we build the base reactive transport model: a voronoi grid, three pumping wells, conservative transport, and then the full PHREEQC reaction network coupled through `mf6rtm`. We finish by running the model once and laying its output alongside the measured field data.

This notebook lives at the start of a strict sequence. The notebooks that follow assume the model and helpers it sets up here:

- [`../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb`](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb) — parameterise the model with `PstFrom`
- [`../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb`](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) — observations, noise, and the synthetic truth
- [`../part1_04_prior_mc/dizon_prior_mc.ipynb`](../part1_04_prior_mc/dizon_prior_mc.ipynb) — the prior Monte Carlo

## What this model is, in two paragraphs

The site, the chemistry and the decision question are told in full in [`../part0_01_intro_to_dizon/intro_to_dizon.ipynb`](../part0_01_intro_to_dizon/intro_to_dizon.ipynb). The short version: at the DIZON site (Someren, NL) oxic, pre-treated surface water is injected ~300 m deep into an anoxic, pyrite-rich aquifer. The oxygen and nitrate in the injectate oxidise pyrite, releasing sulfate and acidity. Reaction rates depend on temperature, so the injectate temperature matters. We want to extract drinking water from a **supply well** (`wellopt`) during a later supply period, and we want to know how much sulfate the supplied water will carry — and how sure we are. Treatment cost scales with that concentration, so the forecast we carry is the *distribution* of peak sulfate (its median and P95), not a single number.

If you have not read part0_01, do that first — this notebook does not re-explain the chemistry. From here on we are mechanics: turning that conceptual model into files MODFLOW 6 and PHREEQC can run.

### The timeline

The simulation is 728 days, an abstraction of the 854-day field experiment (faithful in chemistry, not a reproduction). Three windows structure everything that follows:

| Window | Days | What happens |
|---|---|---|
| **History period** | 0 – 252 | injection runs; the flush well (`wellout`) operates; monitoring data exists and may be used for conditioning |
| **Decision date** | 252 | the operate/treat decision must be committed — note the gap before supply starts |
| **Supply period** | 308 – 728 | the supply well (`wellopt`) extracts drinking water; the forecast (peak SO₄) lives here |

The lead-time gap between the decision date (252) and supply switch-on (308) is deliberate: decisions are made ahead of time, not the morning the pump starts.

### Admin

We start, as always, by loading the libraries we need and the shared helper module `herebedragons` (imported as `hbd`). The build itself leans on convenience functions in that module — `make_chd`, `make_wel_in`, `get_botms`, `make_obs_pack`, `get_bins` and friends — so we do not have to re-type boundary and well plumbing every time.

The model inputs (the domain, well locations, initial chemistry, the heat/borehole data) are pre-prepared and live in the repository's `data/` directory. **We never write generated files into `data/`** — every output of this notebook goes into a workspace folder created inside this notebook's own directory.

In [ ]:
import os
import sys
import shutil
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from shapely import box, Point, LineString
from IPython.display import display

import flopy
import pyemu
from flopy.utils import cvfdutil
from mf6rtm import utils, mup3d
from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator
from vorflow.utils import calculate_mesh_quality, summarize_quality

# use the vendored flopy/pyemu, not whatever is on the system
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__

sys.path.insert(0, "..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10

The model inputs sit in the repository-level `data/` directory, two levels up from this notebook. We point a `data_d` variable at it once; the build functions below take `data_d` as an argument and pass it on to the `herebedragons` helpers, so every input is read through it and the notebook runs correctly from its own directory. We also make the workspace folder this notebook will write into:

In [ ]:
# repository data directory (inputs only - never written to)
data_d = os.path.join("..", "..", "data")
assert os.path.isdir(data_d), f"cannot find the repo data directory at {data_d}"

# this notebook's workspace - everything generated lands here
ws = Path("model")
if ws.exists():
    shutil.rmtree(ws)
ws.mkdir()

## Grid and domain

A few decisions baked into the grid before we begin:

- a **disv voronoi grid**, refined around the wells (so concentration fronts near the wells are resolved without paying for fine cells everywhere);
- **12 layers** vertically, so we can represent depth-dependent redox zonation and screen the monitoring filters at the right depths;
- **35 stress periods** with daily time steps over 728 days;
- a regional aquifer represented by constant-head (`CHD`) boundaries on the left and right edges.

First, read the domain polygon and build a rectangular extent from its bounds:

In [ ]:
domain = gpd.read_file(Path(data_d, 'domain.gpkg'))
Ly = domain.geometry.total_bounds[3] - domain.geometry.total_bounds[1]
Lx = domain.geometry.total_bounds[2] - domain.geometry.total_bounds[0]

xul = domain.geometry.total_bounds[0]
yul = domain.geometry.total_bounds[1]

domain_ext = box(xul, yul, xul + Lx, yul + Ly)
domain_ext

Read the three wells and the refinement polygon. The wells carry their code IDs here — `wellin` (injection), `wellout` (flush) and `wellopt` (supply) — the descriptive names are used everywhere in prose.

In [ ]:
wells = pd.read_csv(Path(data_d, 'wells.csv'))
refinement = gpd.read_file(Path(data_d, 'refinement.gpkg'))

The refinement polygon gives us a line along which to thicken the mesh (the injection-to-extraction transect, where the fronts move):

In [ ]:
geom = refinement.geometry[0]
minx, miny, maxx, maxy = geom.bounds

# lines along the top and bottom edges of the refinement polygon
top_line = LineString([(minx, maxy), (maxx, maxy)])
bottom_line = LineString([(minx, miny), (maxx, miny)])

Now build the conceptual mesh. `background_lc` is the background characteristic length (cell size, in metres) away from any refinement; the wells get a much finer resolution and the transect line an intermediate one. `vorflow` then triangulates and converts to a voronoi grid clipped to the domain.

In [ ]:
background_lc = 150.0  # background cell size (m); wells and the transect are refined below

blueprint = ConceptualMesh()
blueprint.add_polygon(domain_ext, zone_id=1)

blueprint.add_line(bottom_line, line_id="Refinement-Line",
                   resolution=20,
                   dist_max=1 * background_lc)

for wid in wells.index:
    blueprint.add_point(Point(wells.loc[wid, 'x'], wells.loc[wid, 'y']),
                        point_id=f"{wells.loc[wid, 'name']}",
                        resolution=3,
                        dist_max=1.5 * background_lc)
    print(f"Added well {wells.loc[wid, 'name']} at ({wells.loc[wid, 'x']}, {wells.loc[wid, 'y']})")

clean_polys, clean_lines, clean_pts = blueprint.generate()

# triangulate, then convert to a clipped voronoi grid
mesher = MeshGenerator(background_lc=background_lc, verbosity=0)
mesher.generate(clean_polys, clean_lines, clean_pts)

tessellator = VoronoiTessellator(mesher, blueprint, clip_to_boundary=True)
grid_gdf = tessellator.generate()

Have a look at the domain, the grid, and the three pumping wells:

In [ ]:
fig, ax = plt.subplots(1, 1)

grid_gdf.plot(ax=ax, alpha=0.5, edgecolor='k', linewidth=0.2)
domain.dissolve().boundary.plot(ax=ax, alpha=1, edgecolor='r', linewidth=1)

cmap = plt.cm.tab10
colors = cmap(np.linspace(0, 1, len(wells)))
for (i, row), color in zip(wells.iterrows(), colors):
    ax.scatter(row['x'], row['y'], facecolor=color, edgecolors='black',
               marker='o', s=50, label=row['name'], zorder=10)

ax.legend()
ax.set_aspect('equal')
fig.tight_layout()

Check the mesh quality and write the grid out as a shapefile. The shapefile is a **generated artifact**, so it goes into the notebook's workspace (`model/`), not into `data/` — the original model wrote it to `data/`, which is the kind of thing that gets a grid silently committed to version control.

In [ ]:
gdf = calculate_mesh_quality(grid_gdf, calc_ortho=True)
summarize_quality(gdf)

grid_shp = Path(ws, "mf6_grid.shp")
grid_gdf.to_file(grid_shp)

## The model-build functions

We wrap the flow and transport build in two functions, `make_gwf` and `make_gwt`, so we can call them twice: once for a conservative single-tracer model (to sanity-check transport), and once for the full multi-species reactive model. Read through them — the magic numbers are called out in the comments and explained in the markdown that follows.

In [ ]:
def make_gwf(ws, grid_shp, data_d, tracer='Cl', mup3d_m=None):
    flow_modelname = "gwf"

    # 35 stress periods, (perlen, nstp, tsmult); daily steps spanning 728 days
    perioddata = [(2, 2, 1), (4, 4, 1), (4, 4, 1), (4, 4, 1), (7, 7, 1),
                  (7, 7, 1), (7, 7, 1), (7, 7, 1), (14, 14, 1), (14, 14, 1),
                  (15, 15, 1), (13, 13, 1), (14, 14, 1), (14, 14, 1), (14, 14, 1),
                  (21, 21, 1), (35, 35, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1),
                  (28, 28, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1),
                  (35, 35, 1), (35, 35, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1),
                  (35, 35, 1), (35, 35, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1)]
    nper = len(perioddata)

    sim = flopy.mf6.MFSimulation(sim_name=flow_modelname, version='mf6', sim_ws=ws)

    # copy the platform binaries (mf6, libmf6, pestpp-*) into the workspace;
    # get_bins resolves the repo bin/ directory itself, so it works from here
    hbd.get_bins(ws)

    tdis = flopy.mf6.ModflowTdis(sim, pname="tdis", time_units="DAYS",
                                 nper=nper, perioddata=perioddata)

    ims = flopy.mf6.ModflowIms(sim, complexity="complex",
                               outer_dvclose=1e-3, inner_dvclose=1e-3,
                               filename=f"{flow_modelname}.ims")
    sim.register_ims_package(ims, [flow_modelname])

    gwf = flopy.mf6.ModflowGwf(sim, modelname=flow_modelname,
                               model_nam_file=f"{flow_modelname}.nam",
                               exe_name='mf6')

    # build the disv grid from the voronoi shapefile we just wrote
    verts, iverts = cvfdutil.shapefile_to_cvfd(str(grid_shp))
    gridprops = cvfdutil.get_disv_gridprops(verts, iverts, xcyc=None)

    disv = flopy.mf6.ModflowGwfdisv(
        gwf,
        nlay=12,          # 12 layers - depth-resolved redox zonation and filter screens
        ncpl=gridprops['ncpl'],
        nvert=gridprops['nvert'],
        vertices=gridprops['vertices'],
        cell2d=gridprops['cell2d'],
        top=-273.0,       # datum (m): the aquifer top is ~273 m below the reference; botms set below
        botm=0.0,         # placeholder; real layer bottoms are kriged in just below
        filename=f"{flow_modelname}.disv")

    # kriged layer bottoms from the borehole data (<data_d>/botm.gpkg)
    botms = hbd.get_botms(gwf, ws, data_d=data_d)
    disv.botm.set_data(botms)
    disv.set_all_data_external()

    nlay = disv.nlay.get_data()
    ncpl = disv.ncpl.get_data()

    # flat initial heads
    strt = 0.0 * np.ones((nlay, ncpl))
    ic = flopy.mf6.ModflowGwfic(gwf, pname="ic", strt=strt)
    ic.set_all_data_external()

    # k and k33 start at 1.0 everywhere; these become the parameterised fields in part1_02
    npf = flopy.mf6.ModflowGwfnpf(gwf, icelltype=0,
                                  k=np.ones((nlay, ncpl)),
                                  k33=np.ones((nlay, ncpl)))
    npf.set_all_data_external()

    ss = np.ones((nlay, ncpl)) * 1.e-4
    sto = flopy.mf6.ModflowGwfsto(gwf, ss=ss, iconvert=0, transient={0: True})
    sto.set_all_data_external()

    # boundaries and wells (plumbing lives in herebedragons)
    hbd.make_chd(gwf, conservative_tracer=tracer, mup3d_m=mup3d_m, data_d=data_d)
    hbd.make_wel_out(sim, conservative_tracer=tracer, mup3d_m=mup3d_m, data_d=data_d)
    hbd.make_wel_in(sim, conservative_tracer=tracer, mup3d_m=mup3d_m, data_d=data_d)
    hbd.make_wel_opt(sim, conservative_tracer=tracer, mup3d_m=mup3d_m, data_d=data_d)

    # output control
    oc = flopy.mf6.ModflowGwfoc(
        gwf,
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        head_filerecord=[f"{flow_modelname}.hds"],
        budget_filerecord=[f"{flow_modelname}.cbb"],
        printrecord=[("HEAD", "LAST")])

    return gwf, sim

In [ ]:
def make_gwt(sim, data_d, tracer='Cl', mup3d_m=None):

    # transport properties, constant across all layers in the base model
    ne = 0.35           # effective porosity (-); becomes a parameter in part1_02
    long_disp = 0.1     # longitudinal dispersivity (m)
    disp_tr_vert = long_disp * 0.01   # transverse vertical dispersivity (m)
    disp_tr_hor = long_disp * 0.1     # transverse horizontal dispersivity (m)
    diffc = 0           # molecular diffusion coefficient (neglected here)

    gwf = sim.get_model('gwf')

    # one gwt model per transported species; the conservative build has a single tracer
    if mup3d_m is not None and tracer is None:
        components = mup3d_m.components
    else:
        components = [tracer]

    for comp in components:
        print(f"Setting transport for {comp}")
        model_name = comp
        gwt = flopy.mf6.MFModel(sim, model_type="gwt6", modelname=model_name,
                                model_nam_file=f"{model_name}.nam")
        ims = flopy.mf6.ModflowIms(sim, complexity="complex",
                                   outer_dvclose=1e-3, inner_dvclose=1e-3,
                                   filename=f"{model_name}.ims")
        sim.register_ims_package(ims, [model_name])

        # reuse the flow model's grid
        dis = gwf.dis
        nlay = dis.nlay.get_data()
        ncpl = dis.ncpl.get_data()
        disv = flopy.mf6.ModflowGwfdisv(gwt, nlay=nlay, ncpl=ncpl,
                                        nvert=dis.nvert.get_data(),
                                        vertices=dis.vertices.get_data(),
                                        cell2d=dis.cell2d.get_data(),
                                        top=dis.top.get_data(),
                                        botm=dis.botm.get_data(),
                                        filename=f"{model_name}.disv")
        disv.set_all_data_external()

        if tracer is not None:
            # conservative tracer: small uniform background concentration
            strt = 2.540000e-04
        else:
            # reactive build: PHREEQC supplies per-species initial concentrations
            strt = mup3d_m.sconc[comp]
        ic = flopy.mf6.ModflowGwtic(gwt, strt=strt, filename=f"{model_name}.ic")
        ic.set_all_data_external()

        adv = flopy.mf6.ModflowGwtadv(gwt, scheme="tvd")
        adv.set_all_data_external()

        alpha_l = np.ones((nlay, ncpl)) * long_disp
        alpha_th = np.ones((nlay, ncpl)) * disp_tr_hor
        alpha_tv = np.ones((nlay, ncpl)) * disp_tr_vert
        dsp = flopy.mf6.ModflowGwtdsp(gwt, xt3d_off=True,
                                      alh=alpha_l, ath1=alpha_th, atv=alpha_tv,
                                      diffc=diffc, filename=f"{model_name}.dsp")
        dsp.set_all_data_external()

        sourcerecarray = [["welin", "aux", model_name],
                          ["welout", "aux", model_name],
                          ["welopt", "aux", model_name],
                          ["chd", "aux", model_name]]
        ssm = flopy.mf6.ModflowGwtssm(gwt, sources=sourcerecarray,
                                      save_flows=True, print_flows=True,
                                      filename=f"{model_name}.ssm")
        ssm.set_all_data_external()

        if comp == 'Tmp':
            # heat is transported as a retarded tracer. distcoef is the sorption
            # distribution coefficient that turns the retardation factor into the
            # observed heat lag; with a bulk density of 1850 it gives the
            # aquifer's volumetric heat capacity ratio. These heat-exchange
            # parameters are held FIXED (not estimated) - the heat signal is used
            # as a velocity tracer, so its retardation is treated as known site
            # physics rather than another uncertain knob to turn.
            distcoef = np.ones((nlay, ncpl)) * 2.1141E-04
            sorption = "Linear"
            pbulk = 1850          # bulk density (kg/m3), constant across layers
            bulk_density = np.ones((nlay, ncpl)) * pbulk
        else:
            distcoef = None
            sorption = None
            bulk_density = None
        porosity = np.ones((nlay, ncpl)) * ne

        mst = flopy.mf6.ModflowGwtmst(gwt, porosity=porosity,
                                      first_order_decay=None, decay=None,
                                      decay_sorbed=None, sorption=sorption,
                                      bulk_density=bulk_density,
                                      distcoef=distcoef, sp2=None,
                                      filename=f"{model_name}.mst")
        mst.set_all_data_external()

        flopy.mf6.ModflowGwtoc(
            gwt,
            budget_filerecord=f"{model_name}.cbb",
            concentration_filerecord=f"{model_name}.ucn",
            concentrationprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 10, "GENERAL")],
            saverecord=[("CONCENTRATION", "ALL")],
            printrecord=[("CONCENTRATION", "LAST")])

        flopy.mf6.ModflowGwfgwt(sim, exgtype="GWF6-GWT6", exgmnamea='gwf',
                                exgmnameb=model_name, filename=f"{model_name}.gwfgwt")
        hbd.make_obs_pack(gwt, data_d=data_d)

    sim.write_simulation()
    return sim

## Conservative transport first

Before any chemistry, let's check the physics. The functions above can build a transport model the plain FloPy way — a single conservative species, no reactions. Here we make a simulation with one tracer model called `Cl` (chloride) and run it. If you are new to transport simulations, this is the cell to poke at: build it, run it, inspect the workspace.

In [ ]:
gwf, sim = make_gwf(ws, grid_shp, data_d, tracer='Cl', mup3d_m=None)
sim = make_gwt(sim, data_d, tracer='Cl', mup3d_m=None)

pyemu.os_utils.run("mf6", cwd=str(ws))

## Chemistry and the reactive transport model

Now the reaction network. The chemistry inputs live in `data/` — initial aquifer chemistry, injectate chemistry, exchanger and mineral surfaces. We will inspect them as we go.

All of the chemistry setup is wrapped in `initialize_chemistry`. It uses the existing `mf6` simulation plus the `mup3d` class from `mf6rtm` to store the chemistry inputs and write the PHREEQC files for us — think of `mup3d` as FloPy, but for the reaction network.

In [ ]:
def initialize_chemistry(ws, data_d, sim, nlay, ncpl):
    perioddata = sim.tdis.perioddata.get_data()
    nper_model = sim.tdis.nper.get_data()

    # background aquifer chemistry, applied uniformly as the initial solution
    solutionsdf = pd.read_csv(Path(data_d, "ic_aq_chem.csv"), index_col=0)

    # injection chemistry: one column per (stress period, layer)
    injdf = pd.read_csv(Path(data_d, "wellin.csv"), index_col=0)
    injdf = injdf[['layer'] + solutionsdf.index.tolist()].copy()

    frames = []
    for per in injdf.index.unique():
        df = (injdf.loc[per].reset_index()
                    .drop(columns='kper')
                    .groupby('layer').mean()
                    .T)
        df.columns = [f"{per}_{layer}" for layer in df.columns]
        frames.append(df)

    injdf = pd.concat(frames, axis=1)
    solutionsdf = pd.concat([solutionsdf, injdf], axis=1)

    # mup3d wants solutions as a dict keyed 1..n; utils does the conversion
    solutions = utils.solution_df_to_dict(solutionsdf)

    # an array the shape of the grid holding the solution id for each cell;
    # all ones = the single background solution everywhere
    sol_ic = np.ones((nlay, ncpl), dtype=float)

    solution = mup3d.Solutions(solutions)
    solution.set_ic(sol_ic)

    # cation exchanger: per-layer exchange-site loadings, renamed to PHREEQC species
    excdf = pd.read_csv(Path(data_d, "ic_exchanger.csv"), comment='#')
    ex_names = {"Ca_ex": "CaX2", "Fe_ex": "FeX2", "K_ex": "KX",
                "Mg_ex": "MgX2", "Na_ex": "NaX"}
    excdf['name'] = excdf['var'].map(ex_names)
    excdf['layer'] -= 1  # zero-index
    excdf = excdf.pivot(index="name", columns="layer", values="value")
    exchanger_dict = excdf.to_dict()
    for k, subdict in exchanger_dict.items():
        for key in subdict:
            subdict[key] = {'m0': subdict[key]}

    exchanger = mup3d.ExchangePhases(exchanger_dict)
    exchanger.set_ic(sol_ic)
    # equilibrate the exchanger against the background solution (id 1) in every layer
    exchanger.set_equilibrate_solutions([1] * nlay)

    # mineral surfaces: convert bulk-volume concentrations to per-water-volume
    mindf = pd.read_csv(Path(data_d, "ic_surfaces.csv"), comment='#')
    mindf['value'] = [utils.concentration_volbulk_to_volwater(i, 0.35)
                      for i in mindf['value'].values]
    mindf = mindf.pivot(index="var", columns="layer", values="value")

    # ferrihydrite and organic matter held at equilibrium; saturation index 0
    eq_m0 = utils.solution_df_to_dict(mindf.loc[['Ferrihydrite', "Orgmatter"], :])
    si = 0  # initial saturation index, following the original model

    eq_dic = {}
    for ly in range(nlay):
        for key in eq_m0.keys():
            eq_dic[ly] = {key: {}}
            eq_dic[ly][key]['si'] = si
            eq_dic[ly][key]['m0'] = eq_m0[key][ly]
    equilibriums = mup3d.EquilibriumPhases(eq_dic)
    equilibriums.set_ic(sol_ic)

    # pyrite is KINETIC (rate-limited) - the case's signature reaction
    py_m0 = utils.solution_df_to_dict(mindf.loc[["Pyrite"], :])

    # pyrite rate-law parameters passed to the PHREEQC RATES block, in order:
    #   [log10 rate constant, reactive-surface exponent, O2 reaction order,
    #    NO3 reaction order]. These tune how fast pyrite oxidises against the
    #    available oxidants and temperature; they are candidate parameters
    #    for part1_02 (the reaction tier of the layered prior).
    kin_py_params = [1.600000e+01, 6.700000e-01, 5.000000e-01, -1.100000e-01]
    kin_dic = {}

    # organic carbon is also kinetic but a minor redox contributor (see below)
    kin_orgc_params = [1.570000e-09, 1.670000e-11, 1.000000e-13]
    orgc_form = "Orgc -1.0 CH2O 1.0"          # PHREEQC stoichiometry for the custom Orgc phase
    orgc_steps = "8.640000e+04 in 1 steps"    # integrate the rate over 86400 s (1 day) in a single step

    for ly in range(nlay):
        for key in py_m0.keys():
            kin_dic[ly] = {key: {}}
            kin_dic[ly][key]['m0'] = py_m0[key][ly]
            kin_dic[ly][key]['parms'] = kin_py_params

    # add organic carbon with m0 = 1.0 in every layer
    for key in kin_dic.keys():
        kin_dic[key]['Orgc'] = {}
        kin_dic[key]['Orgc']['m0'] = 1.0
        kin_dic[key]['Orgc']['parms'] = kin_orgc_params
        kin_dic[key]['Orgc']['formula'] = orgc_form
        kin_dic[key]['Orgc']['steps'] = orgc_steps

    kinetics = mup3d.KineticPhases(kin_dic)
    kinetics.set_ic(sol_ic)

    model = mup3d.Mup3d('model', solution, nlay=nlay, ncpl=ncpl)
    model.set_wd(ws)

    # PHREEQC thermodynamic database and the postfix block, copied/pointed from data/
    shutil.copy(Path(data_d, 'datab.dat'), Path(model.wd, 'datab.dat'))
    model.set_database(Path('datab.dat'))
    model.set_postfix(Path(data_d, 'postfix.phqr'))

    model.set_exchange_phases(exchanger)
    model.set_phases(kinetics)
    model.set_phases(equilibriums)

    # run reactions only at the (kper, kstp) pairs we ask for - here, every 7 days
    tsteps = hbd.create_reactive_tsteps(perioddata, output_interval=7)
    model.set_config(reactive={"timing": 'user',
                               "externalio": True,
                               "tsteps": tsteps})
    model.set_componenth2o(True)
    model.initialize(add_charge_flag=False)
    return model

Take a moment with the function above. A few things worth flagging:

- **pyrite is kinetic, the rest is equilibrium.** Pyrite oxidation is rate-limited and temperature-dependent — that is the whole story of the site — so it gets a rate law. Ferrihydrite and organic matter are handled at equilibrium.
- **organic matter is included but minor.** The source study found organic carbon a small redox contributor compared with pyrite; it is kept in the network for fidelity (`m0 = 1.0` per layer) but, as the layered prior in part1_02 will show, it is *not* an estimated parameter — pyrite carries the reaction uncertainty.
- **reactions run every 7 days** (`output_interval=7`), not every time step. Running PHREEQC on every cell every day would be far slower; the 7-day cadence is a cost/accuracy compromise.

Now load the conservative simulation back in to get the grid dimensions, then initialise the chemistry:

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=str(ws), sim_name='gwf',
                                  version='mf6', exe_name='mf6',
                                  verbosity_level=0)
gwf = sim.get_model("gwf")
nlay = gwf.dis.nlay.get_data()
ncpl = gwf.dis.ncpl.get_data()

# clear the workspace - we re-write everything for the multi-species model below
if ws.exists():
    shutil.rmtree(ws)

mup3d_m = initialize_chemistry(ws, data_d, sim, nlay, ncpl)

Up to here, only the chemistry files are written. The print-out tells you which external files were written (exchange, equilibrium and kinetic phases) and at which stress periods and time steps reactions will be invoked.

Next we re-write the `mf6` files, this time for the full multi-species simulation — one transport model per chemical component, sources tagged so PHREEQC can read them.

In [ ]:
gwf, sim = make_gwf(ws, grid_shp, data_d, tracer=None, mup3d_m=mup3d_m)
sim = make_gwt(sim, data_d, tracer=None, mup3d_m=mup3d_m)

## Feel the cost: one base-model run

We are ready to run the base reactive transport model. **Brace yourself — this takes about 6 minutes on macOS or Linux, and 10–15 minutes on Windows.** Reactive transport models are slow: PHREEQC is solving a stiff chemical system in every reactive cell at every reactive step.

That ~6 min/run is not a detail. It is the reason this whole curriculum exists. A history match or a risk analysis needs *hundreds* of runs — at 6 minutes each, a 201-realisation ensemble is the better part of a day of compute, and that is before any iteration. This single run is your calibration of that cost; keep it in mind every time we reach for an ensemble. From part1_05 onward we lean on **emulation (DSI)** precisely so we do not pay this bill hundreds of times over.

Run it once, now, so the number is real to you:

In [ ]:
pyemu.os_utils.run("mf6rtm", cwd=str(ws))

## Measured data: a credibility check, not conditioning

Before the PEST machinery, let's look at the measured field data. All observations live in:

`../../data/obs_chem_cleaned.csv`

In a rare win for a tutorial, we have **real measured data** — from four monitoring sites plus the injection/extraction wells, across the simulation period, each site potentially screened at several depths.

A word on what we do and do not do with it. Most of this curriculum works against a **synthetic truth**: a single realisation drawn from the prior ensemble, declared to be "reality", and held out (we pick it in part1_03). The synthetic truth, not the measured data, drives history matching, posterior scoring and emulator-fidelity checks. So why look at the real data at all? Because comparing the base model against it tells us whether the model is *credible* — whether it reproduces the right shapes and magnitudes — before we trust it for anything. This is a sanity check on the conceptual model, **not** conditioning. (If you are feeling adventurous, the optional real-data capstone history-matches against the measured data directly.)

One quirk: some monitoring sites in the CSV have empty rows. Those are placeholders — they define observation *locations* we will track with PEST++ later (the supply-well sites among them), even where no field measurement exists.

In [ ]:
meas_data = pd.read_csv(Path(data_d, "obs_chem_cleaned.csv"))
meas_data.head()

Concentrations are in mol/L (molar) — and conveniently, so is the default `mf6rtm` output. `obsid` names the monitoring site, `variable` the species or property, `time` the day, and `x`/`y`/`layer` the location.

Have a look at the sites and variables on offer:

In [ ]:
display(meas_data.obsid.unique())
display(meas_data.variable.unique())

That is a lot of sites and a lot of chemistry. If you come from flow modelling, where you mostly juggle heads and a flux or two, this can feel like a lot.

To keep the curriculum tractable we focus on **three monitoring sites** and a handful of species. The sites:

- `wp1-f3` — a near-well monitoring filter
- `wp4-f5` — a shallow filter further along the transect
- `welopt-ly3` — the **supply well** (`wellopt`), where the forecast lives

and the species we lean on for conditioning are sulfate (SO₄, the forecast species), oxygen (O0) and nitrate (NO3) (the oxidant-consumption signal), pH (buffering) and temperature (Tmp, the heat tracer). The major cations are deliberately held back for now — we come back to them as a dataworth question later. (The supply-well obsids come in three layers, `welopt-ly1/ly3/ly5`; we use `welopt-ly3` consistently from here on.)

Let's line up simulated and measured outputs.

## Inspecting the base model

The model is built and run. Now the part many modellers enjoy: poking at the output and checking it against your conceptualisation. Lining up simulated and observed series at the monitoring points is a good way to confirm the grid, layers and boundaries are doing what you expect — and to see where the model captures the data and where it does not.

The helper `process_sim_conc` does the matching: for each observation location, layer and variable it pairs simulated and measured values at each time, ready to plot. It reads the observation CSV from inside the workspace, so first copy it in.

In [ ]:
# bring a copy of the measured data into the workspace for the post-processor
shutil.copy(Path(data_d, "obs_chem_cleaned.csv"), Path(ws, "obs_chem_cleaned.csv"))

fname, obs = hbd.process_sim_conc(wd=str(ws))

Load the matched table and plot simulated against measured for the focus sites. `1e30` is MODFLOW's no-data flag, so mask it out first. Use the dropdown to step through the variables — start with SO₄, the species the whole forecast turns on.

In [ ]:
obs = pd.read_csv(Path(ws, fname))
cols = ["sim", "meas"]
obs[cols] = obs[cols].mask(obs[cols] >= 1e30, np.nan)

obs_to_plot = ['welopt-ly3', 'wp1-f3', 'wp4-f5']
variables = sorted(obs["variable"].unique())

def plot_var(var_to_plot, measured_data=True):
    fig, axs = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(8, 2))
    c = 1e3 if var_to_plot == 'Tmp' else 1   # Tmp is stored scaled by 1e-3
    for ax, oid in zip(axs.flatten(), obs_to_plot):
        tmp = obs[(obs.obsid == oid) & (obs.variable == var_to_plot)]
        ax.plot(tmp.time, tmp.sim * c, label="sim")
        if measured_data:
            ax.scatter(tmp.time, tmp.meas * c, s=15, label="meas")
        ax.set_title(oid.upper())
        ax.set_ylabel(var_to_plot)
    axs[0].legend()
    fig.tight_layout()
    plt.show()

dropdown = widgets.Dropdown(options=variables, value="SO4", description="Variable:")
widgets.interact(plot_var, var_to_plot=dropdown);

How does it look? The base model should track the broad behaviour of the measured data — the timing and magnitude of the sulfate rise, the oxygen and nitrate drawdown, the temperature signal — without nailing every point. That is what we want at this stage: a credible model, not a calibrated one. The mismatch you see now is exactly the gap the rest of the curriculum sets out to characterise.

For reference, here are the cumulative end-of-period days — the stress-period boundaries that mark the history period (ends at day 252), the decision date (252) and the supply period (308–728):

In [ ]:
sim.tdis.perioddata.get_data().perlen.cumsum()

Finally, a map: simulated heads early in the simulation, the wells, the constant-head boundaries (green), and the monitoring sites. This is the planimetric companion to the conceptual model — a last check that the flow field and the observation network sit where we think they do.

In [ ]:
xc = gwf.modelgrid.xcellcenters
yc = gwf.modelgrid.ycellcenters

# map each obsid to its grid cell centre, then collapse to one point per site
obs.cell2d = obs.cell2d.astype(int)
obs['x'] = obs.cell2d.apply(lambda c: xc[c])
obs['y'] = obs.cell2d.apply(lambda c: yc[c])
obscoords = obs[['obsid', 'x', 'y']].drop_duplicates().copy()
obscoords['obsid'] = obscoords.obsid.str.split("-").str[0]
obscoords.drop_duplicates(subset='obsid', inplace=True)
obscoords

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
pm = flopy.plot.PlotMapView(gwf, ax=ax)

grid_gdf.plot(ax=ax, alpha=0.5, edgecolor='k', linewidth=0.2)
domain.dissolve().boundary.plot(ax=ax, alpha=1, edgecolor='r', linewidth=1)

arr = gwf.output.head().get_data(totim=50)
vmin, vmax = -7.5, 5
levels = np.linspace(vmin, vmax, 6)
levels = levels[levels != 0]
pa = pm.plot_array(arr, masked_values=[1e30], cmap='Blues', alpha=0.7, vmin=vmin, vmax=vmax)
pclines = pm.contour_array(arr, masked_values=[1e30], colors='k', linewidths=0.5, levels=levels)
plt.clabel(pclines, inline=True, fontsize=8, fmt="%.1f")
cb = plt.colorbar(pa, ax=ax, shrink=0.5)
cb.set_label("Head (m)")

# injection and flush wells
colors = ['b', 'r']
for (i, row), color in zip(wells.loc[wells.name != 'wellopt'].iterrows(), colors):
    ax.scatter(row['x'], row['y'], facecolor=color, edgecolors='black',
               marker='o', s=50, label=row['name'], zorder=10)

# monitoring sites
ax.scatter(obscoords.x, obscoords.y, c='k', marker='x')
for oid, row in obscoords.iterrows():
    ax.annotate(row['obsid'], (row['x'], row['y']), textcoords="offset points",
                xytext=(0, 5), ha='center', fontsize=8)

# constant-head boundaries on the left and right edges
xmin, xmax = ax.get_xlim()
ax.vlines(x=xmin, ymin=0, ymax=ax.get_ylim()[-1], colors='g', linestyles='-', lw=5, label="CHD", zorder=50)
ax.vlines(x=xmax, ymin=0, ymax=ax.get_ylim()[-1], colors='g', linestyles='-', lw=5, zorder=50)

ax.legend()
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_aspect('equal')
fig.tight_layout()

That is the base model: built, run once (so the cost is real to you), and checked against the measured data for credibility. Nothing here is calibrated or conditioned — that work starts next.

Continue to [`../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb`](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb), where we wrap this model in PEST++ and build the layered prior.